In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
!pip install selenium webdriver-manager beautifulsoup4 lxml pandas -q

# Vérifier si Chrome est déjà installé
import subprocess, shutil
chrome_ok = shutil.which('google-chrome') or shutil.which('google-chrome-stable') or shutil.which('chromium-browser')
if not chrome_ok:
    print('Installation de Google Chrome...')
    !apt-get update -q
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !dpkg -i google-chrome-stable_current_amd64.deb
    !apt-get install -f -y -q
else:
    print(f'Chrome déjà installé : {chrome_ok}')

# Vérifier la version
result = subprocess.run(['google-chrome', '--version'], capture_output=True, text=True)
print('Version Chrome :', result.stdout.strip())

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime

print("✅ Imports OK")

In [ ]:
# ── Configuration du driver Chrome ────────────────────────────────────────────
import subprocess, re as _re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_chrome_binary():
    """Retourne le chemin vers l'exécutable Chrome disponible."""
    import shutil
    for bin_name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
        path = shutil.which(bin_name)
        if path:
            return path
    return None

def creer_driver():
    options = Options()
    options.add_argument('--headless=new')           # nouvelle syntaxe headless
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-extensions')
    options.add_argument('--remote-debugging-pipe')  # évite le port conflict
    options.add_argument('user-agent=Mozilla/5.0 (X11; Linux x86_64) '
                         'AppleWebKit/537.36 (KHTML, like Gecko) '
                         'Chrome/120.0.0.0 Safari/537.36')
    # Pointe vers le bon binaire Chrome
    chrome_bin = get_chrome_binary()
    if chrome_bin:
        options.binary_location = chrome_bin

    # webdriver-manager télécharge le ChromeDriver compatible automatiquement
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(30)
    return driver

# Test rapide
print('Test du driver...')
try:
    d = creer_driver()
    d.get('https://www.google.com')
    print('✅ Driver Chrome OK :', d.title)
    d.quit()
except Exception as e:
    print('❌ Erreur driver :', e)

In [ ]:
# ── Utilitaires de nettoyage ──────────────────────────────────────────────────

VILLES_SENEGAL = [
    "Dakar", "Thiès", "Saint-Louis", "Ziguinchor", "Kaolack", "Touba",
    "Mbour", "Diourbel", "Tambacounda", "Kolda", "Louga", "Fatick",
    "Kaffrine", "Kédougou", "Sédhiou", "Matam", "Pikine", "Rufisque",
    "Guédiawaye", "Tivaouane", "Joal", "Bargny",
]

CONTRATS = ["CDI", "CDD", "Stage", "Freelance", "Temps partiel",
            "Intérim", "Alternance", "Consultant", "Bénévolat", "Vacataire"]

COMPETENCES_KEYWORDS = [
    "Python", "Java", "JavaScript", "SQL", "Excel", "Power BI", "Tableau",
    "R", "SPSS", "SAS", "Machine Learning", "Deep Learning", "NLP",
    "Django", "Flask", "React", "Angular", "Node.js", "PHP", "Laravel",
    "C\+\+", "C#", "Kotlin", "Swift", "Dart", "Flutter",
    "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Git",
    "Photoshop", "Illustrator", "Figma", "InDesign",
    "Gestion de projet", "Agile", "Scrum", "ITIL", "PMP",
    "SAP", "Oracle", "ERP", "CRM", "Salesforce",
    "Comptabilité", "Audit", "Finance", "Fiscalité", "RH",
    "Marketing digital", "SEO", "SEM", "Community management",
    "Anglais", "Français", "Wolof", "Communication", "Négociation",
    "Analyse de données", "Statistiques", "Recherche",
]

MOIS_FR = {
    "janvier": "01", "février": "02", "mars": "03", "avril": "04",
    "mai": "05", "juin": "06", "juillet": "07", "août": "08",
    "septembre": "09", "octobre": "10", "novembre": "11", "décembre": "12",
    "jan": "01", "fév": "02", "avr": "04", "juil": "07", "aoû": "08",
    "sep": "09", "oct": "10", "nov": "11", "déc": "12",
}

def nettoyer_date(texte):
    """Tente de parser une date dans de nombreux formats."""
    if not texte:
        return pd.NaT
    texte = str(texte).strip().lower()

    # Format ISO : 2024-12-05 ou 2024-12-05T10:30:00
    m = re.search(r"(\d{4}-\d{2}-\d{2})", texte)
    if m:
        try:
            return pd.to_datetime(m.group(1))
        except:
            pass

    # Format français : 05 décembre 2024 / 5 déc. 2024
    for mois_fr, mois_num in MOIS_FR.items():
        pattern = rf"(\d{{1,2}})\s+{mois_fr}\.?\s+(\d{{4}})"
        m = re.search(pattern, texte)
        if m:
            try:
                return pd.to_datetime(f"{m.group(2)}-{mois_num}-{m.group(1).zfill(2)}")
            except:
                pass

    # Format court : 05/12/2024 ou 05-12-2024
    m = re.search(r"(\d{2})[/-](\d{2})[/-](\d{4})", texte)
    if m:
        try:
            return pd.to_datetime(f"{m.group(3)}-{m.group(2)}-{m.group(1)}")
        except:
            pass

    # Fallback pandas
    try:
        return pd.to_datetime(texte, dayfirst=True)
    except:
        return pd.NaT


def extraire_ville(texte, soup=None):
    """Cherche une ville sénégalaise dans le texte avec plusieurs stratégies."""
    if not texte:
        return ""

    # Stratégie 1 : recherche directe des villes connues
    for ville in VILLES_SENEGAL:
        if re.search(rf"\b{ville}\b", texte, re.IGNORECASE):
            return ville.title()

    # Stratégie 2 : pattern "à [Ville]"
    m = re.search(r"\bà\s+([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ\-]+(?:\s+[A-ZÀ-Ÿ][a-zA-ZÀ-ÿ\-]+)?)", texte)
    if m:
        candidate = m.group(1).strip()
        if len(candidate) >= 3 and not candidate.lower() in ["le", "la", "les", "un", "une"]:
            return candidate

    # Stratégie 3 : chercher "Localisation" ou "Lieu" dans le HTML structuré
    if soup:
        for label in soup.find_all(["dt", "th", "label", "strong", "b"]):
            txt = label.get_text(strip=True).lower()
            if any(k in txt for k in ["lieu", "localis", "ville", "location"]):
                sibling = label.find_next_sibling() or label.find_next(["dd", "td", "span", "p"])
                if sibling:
                    return sibling.get_text(strip=True)
    return ""


def extraire_contrat(texte, soup=None):
    """Extrait le type de contrat avec plusieurs stratégies."""
    if not texte:
        return ""

    # Stratégie 1 : recherche des mots-clés connus
    pattern = r"\b(" + "|".join(CONTRATS) + r")\b"
    m = re.search(pattern, texte, re.IGNORECASE)
    if m:
        return m.group(1).upper() if m.group(1).upper() in ["CDI", "CDD"] else m.group(1).title()

    # Stratégie 2 : chercher dans les balises structurées
    if soup:
        for label in soup.find_all(["dt", "th", "label", "strong", "b"]):
            txt = label.get_text(strip=True).lower()
            if any(k in txt for k in ["contrat", "type de poste", "type d'emploi", "catégorie"]):
                sibling = label.find_next_sibling() or label.find_next(["dd", "td", "span", "p"])
                if sibling:
                    return sibling.get_text(strip=True)
    return ""


def extraire_entreprise(texte, soup, description=""):
    """Extrait le nom de l'entreprise avec plusieurs stratégies."""

    # Stratégie 1 : balises structurées HTML
    if soup:
        for label in soup.find_all(["dt", "th", "label", "strong", "b"]):
            txt = label.get_text(strip=True).lower()
            if any(k in txt for k in ["entreprise", "société", "recruteur", "employeur", "company"]):
                sibling = label.find_next_sibling() or label.find_next(["dd", "td", "span", "a", "p"])
                if sibling:
                    val = sibling.get_text(strip=True)
                    if val and len(val) > 1:
                        return val

        # Balise meta "author" ou og:site_name
        for meta in soup.find_all("meta"):
            name = meta.get("name", "") + meta.get("property", "")
            if "author" in name.lower() or "site_name" in name.lower():
                val = meta.get("content", "")
                if val and "goafrica" not in val.lower() and "senjob" not in val.lower():
                    return val.strip()

    # Stratégie 2 : pattern "X recrute" dans la description
    if description:
        m = re.search(r"^([A-Za-zÀ-ÿ0-9\s\-&\.,']+?)\s+recrute", description)
        if m:
            candidate = m.group(1).strip()
            if len(candidate) <= 60:
                return candidate

    # Stratégie 3 : pattern "Société : X"
    m = re.search(r"(?:société|entreprise|recruteur)\s*:\s*([A-Za-zÀ-ÿ0-9\s\-&\.]+)",
                  texte, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    return ""


def extraire_competences(texte):
    """Recherche les compétences techniques et transversales mentionnées."""
    if not texte:
        return ""
    trouvees = []
    for comp in COMPETENCES_KEYWORDS:
        if re.search(rf"\b{comp}\b", texte, re.IGNORECASE):
            trouvees.append(comp.replace("\\", ""))
    return ", ".join(trouvees) if trouvees else ""


def extraire_date_html(soup, texte_complet):
    """Cherche la date de publication en cascade dans le HTML."""

    # 1. Balise <time datetime=...>
    time_tag = soup.find("time", {"datetime": True})
    if time_tag:
        val = time_tag.get("datetime", "").strip()
        parsed = nettoyer_date(val)
        if not pd.isnull(parsed):
            return parsed

    # 2. Balise <time> sans datetime
    time_tag = soup.find("time")
    if time_tag:
        val = time_tag.get_text(strip=True)
        parsed = nettoyer_date(val)
        if not pd.isnull(parsed):
            return parsed

    # 3. Meta tag og:article:published_time
    for meta in soup.find_all("meta"):
        prop = meta.get("property", "") + meta.get("name", "")
        if "published" in prop.lower() or "date" in prop.lower():
            val = meta.get("content", "")
            parsed = nettoyer_date(val)
            if not pd.isnull(parsed):
                return parsed

    # 4. Recherche dans les labels structurés (ex: "Publié le :")
    if soup:
        for label in soup.find_all(["dt", "th", "label", "strong", "b", "span"]):
            txt = label.get_text(strip=True).lower()
            if any(k in txt for k in ["publié", "date de publication", "posted", "date d'ajout"]):
                sibling = label.find_next_sibling() or label.find_next(["dd", "td", "span", "p"])
                if sibling:
                    parsed = nettoyer_date(sibling.get_text(strip=True))
                    if not pd.isnull(parsed):
                        return parsed

    # 5. Regex dans le texte brut
    patterns_dates = [
        r"Publié[e]?\s*(?:le)?\s*:\s*(\d{1,2}\s+\w+\.?\s*\d{4})",
        r"Posté[e]?\s*(?:le)?\s*:\s*(\d{1,2}\s+\w+\.?\s*\d{4})",
        r"Date\s*(?:de\s*publication)?\s*:\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        r"(\d{1,2}\s+(?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\s+\d{4})",
    ]
    for pat in patterns_dates:
        m = re.search(pat, texte_complet, re.IGNORECASE)
        if m:
            parsed = nettoyer_date(m.group(1))
            if not pd.isnull(parsed):
                return parsed

    return pd.NaT


print("✅ Utilitaires chargés")

In [ ]:
# ── Scraper GoAfricaOnline ────────────────────────────────────────────────────
def scraper_goafrica(max_pages=150):
    driver = creer_driver()
    offres = []
    tous_les_liens = []

    try:
        # ── ÉTAPE 1 : Collecter tous les liens ──
        for page in range(1, max_pages + 1):
            url = f"https://www.goafricaonline.com/sn/emploi?page={page}"
            try:
                driver.get(url)
                time.sleep(3)
            except Exception as e:
                print(f"  ⚠️ Erreur chargement page {page}: {e}")
                continue

            soup = BeautifulSoup(driver.page_source, "lxml")
            liens_page = []

            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "/sn/emploi/job-" in href:
                    lien = ("https://www.goafricaonline.com" + href
                            if href.startswith("/") else href)
                    if lien not in tous_les_liens:
                        tous_les_liens.append(lien)
                        liens_page.append(lien)

            print(f"[GoAfrica] Page {page} → {len(liens_page)} liens "
                  f"(total: {len(tous_les_liens)})")

            if not liens_page:
                print(f"[GoAfrica] Plus de nouvelles offres à la page {page}, arrêt.")
                break

        print(f"\n✅ Total liens GoAfrica : {len(tous_les_liens)}")

        # ── ÉTAPE 2 : Extraire les données de chaque offre ──
        for i, lien in enumerate(tous_les_liens):
            try:
                driver.get(lien)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, "lxml")
                body_text = soup.get_text(" ", strip=True)

                # ── Titre ──
                titre = ""
                h1 = soup.find("h1")
                if h1:
                    titre = h1.get_text(strip=True)
                if not titre and soup.title:
                    titre = re.sub(r"\s*[\|\-–]\s*Go Africa.*$", "",
                                   soup.title.string or "").strip()

                if not titre:
                    continue  # Offre inutilisable sans titre

                # ── Meta description ──
                meta_desc = soup.find("meta", {"name": "description"})
                description = meta_desc["content"].strip() if meta_desc else ""

                # ── Données ──
                ville      = extraire_ville(body_text, soup) or extraire_ville(description)
                contrat    = extraire_contrat(body_text, soup)
                entreprise = extraire_entreprise(body_text, soup, description)
                competences = extraire_competences(body_text)
                date_pub   = extraire_date_html(soup, body_text)

                offres.append({
                    "titre":            titre,
                    "entreprise":       entreprise,
                    "ville":            ville,
                    "contrat":          contrat,
                    "date_publication": date_pub,
                    "competences":      competences,
                    "lien":             lien,
                    "source":           "goafricaonline.com",
                })

                if (i + 1) % 25 == 0 or i == 0:
                    print(f"  [{i+1}/{len(tous_les_liens)}] ✓ {titre[:55]}")

            except Exception as e:
                print(f"  ✗ Erreur sur {lien}: {e}")

    finally:
        driver.quit()

    return offres

print("✅ Scraper GoAfrica prêt")

In [ ]:
# ── Scraper SenJob ────────────────────────────────────────────────────────────
def scraper_senjob(max_pages=8):
    driver = creer_driver()
    offres = []
    tous_les_liens = []

    try:
        # ── ÉTAPE 1 : Collecter tous les liens ──
        for page in range(1, max_pages + 1):
            url = f"https://senjob.com/sn/offres-d-emploi.php?page={page}"
            try:
                driver.get(url)
                time.sleep(4)
            except Exception as e:
                print(f"  ⚠️ Erreur chargement page {page}: {e}")
                continue

            soup = BeautifulSoup(driver.page_source, "lxml")
            liens_page = []

            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "/sn/jobseekers/" in href and "_e_" in href and href.endswith(".html"):
                    lien = href if href.startswith("http") else "https://senjob.com" + href
                    if lien not in tous_les_liens:
                        tous_les_liens.append(lien)
                        liens_page.append(lien)

            print(f"[SenJob] Page {page} → {len(liens_page)} liens "
                  f"(total: {len(tous_les_liens)})")

            if not liens_page:
                print(f"[SenJob] Fin à la page {page}.")
                break

        print(f"\n✅ Total liens SenJob : {len(tous_les_liens)}")

        # ── ÉTAPE 2 : Extraire les données de chaque offre ──
        for i, lien in enumerate(tous_les_liens):
            try:
                driver.get(lien)
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, "lxml")
                body_text = soup.get_text(" ", strip=True)

                # ── Titre ──
                titre = ""
                h1 = soup.find("h1")
                if h1:
                    titre = h1.get_text(strip=True)
                if not titre and soup.title:
                    titre = re.sub(r"\s*[\|\-–]\s*Senjob.*$", "",
                                   soup.title.string or "",
                                   flags=re.IGNORECASE).strip()
                if not titre:
                    continue

                # ── Meta description ──
                meta_desc = soup.find("meta", {"name": "description"})
                description = meta_desc["content"].strip() if meta_desc else ""

                # ── Données ──
                ville       = extraire_ville(body_text, soup) or extraire_ville(description)
                contrat     = extraire_contrat(body_text, soup)
                entreprise  = extraire_entreprise(body_text, soup, description)
                competences = extraire_competences(body_text)
                date_pub    = extraire_date_html(soup, body_text)

                offres.append({
                    "titre":            titre,
                    "entreprise":       entreprise,
                    "ville":            ville,
                    "contrat":          contrat,
                    "date_publication": date_pub,
                    "competences":      competences,
                    "lien":             lien,
                    "source":           "senjob.com",
                })

                if (i + 1) % 10 == 0 or i == 0:
                    print(f"  [{i+1}/{len(tous_les_liens)}] ✓ {titre[:55]}")

            except Exception as e:
                print(f"  ✗ Erreur : {e}")

    finally:
        driver.quit()

    return offres

print("✅ Scraper SenJob prêt")

In [ ]:
# ── Lancement des scrapers ────────────────────────────────────────────────────
print("=" * 60)
print("=== SCRAPING SENJOB ===")
print("=" * 60)
offres_senjob = scraper_senjob(max_pages=8)
print(f"\nSenJob collecté : {len(offres_senjob)} offres")

print()
print("=" * 60)
print("=== SCRAPING GOAFRICAONLINE ===")
print("=" * 60)
offres_goafrica = scraper_goafrica(max_pages=150)
print(f"\nGoAfricaOnline collecté : {len(offres_goafrica)} offres")

=== SCRAPING SENJOB ===
[SenJob] Page 1 → 43 liens (total: 43)
[SenJob] Page 2 → 39 liens (total: 82)
[SenJob] Page 3 → 36 liens (total: 118)
[SenJob] Page 4 → 38 liens (total: 156)
[SenJob] Page 5 → 32 liens (total: 188)
[SenJob] Page 6 → 33 liens (total: 221)
[SenJob] Page 7 → 0 liens (total: 221)
[SenJob] Fin à la page 7.

✅ Total liens SenJob : 221
  [1/221] ✓ Dev Full Stack Expérimenté – Java / Angular
  [10/221] ✓ Télévendeurs en conquête mobile/marché français
  [20/221] ✓ Professeur Français Orphelinat
  [30/221] ✓ Consultant Support FAI avec expertise N1
  [40/221] ✓ Chargé des centrales d’information
  [50/221] ✓ Responable Pays
  [60/221] ✓ Responsable Administratif et Financier
  [70/221] ✓ Livraison/ Vente moto
  [80/221] ✓ Conducteur d’Engins
  [90/221] ✓ Commercial terrain
  [100/221] ✓ Commercial Terrain
  [110/221] ✓ Attaché(e) Commercial(e)
  [120/221] ✓ Commissioning Engineer
  [130/221] ✓ Chauffeur VTC
  [140/221] ✓ Stagiaire Archiviste Financier
  [150/221] ✓ Manag

In [ ]:
# ── Fusion, nettoyage & rapport qualité ──────────────────────────────────────
df = pd.concat(
    [pd.DataFrame(offres_senjob), pd.DataFrame(offres_goafrica)],
    ignore_index=True
)

# Supprimer les doublons exacts
df = df.drop_duplicates(subset=["titre", "lien"])

# Supprimer les lignes sans titre
df = df[df["titre"].str.strip() != ""]

# Remplacer les chaînes vides par NaN pour le rapport
df_rapport = df.copy()
df_rapport.replace("", pd.NA, inplace=True)

# Conserver les chaînes vides comme vides dans le fichier final
# mais marquer les champs vraiment manquants
df["date_publication"] = df["date_publication"].apply(
    lambda x: nettoyer_date(x) if not isinstance(x, pd.Timestamp) else x
)

# ── Rapport qualité ──
print("=" * 60)
print("BILAN SCRAPING")
print("=" * 60)
print(f"SenJob        : {len(offres_senjob):>5} offres")
print(f"GoAfricaOnline: {len(offres_goafrica):>5} offres")
print(f"Total fusionné: {len(df):>5} offres")

print()
print("── Taux de remplissage par colonne ──")
cols_cibles = ["titre", "entreprise", "ville", "contrat", "date_publication", "competences"]
for col in cols_cibles:
    n_rempli = df_rapport[col].notna().sum()
    pct = n_rempli / len(df) * 100 if len(df) > 0 else 0
    bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    print(f"  {col:<20} [{bar}] {pct:5.1f}%  ({n_rempli}/{len(df)})")

print()
print("── Top villes ──")
print(df[df["ville"] != ""]["ville"].value_counts().head(10))

print()
print("── Répartition contrats ──")
print(df[df["contrat"] != ""]["contrat"].value_counts())

print()
print("── Répartition par source ──")
print(df["source"].value_counts())

BILAN SCRAPING
SenJob        :   221 offres
GoAfricaOnline:  4167 offres
Total fusionné:  4388 offres

── Taux de remplissage par colonne ──
  titre                [████████████████████] 100.0%  (4388/4388)
  entreprise           [███████████████████░]  95.2%  (4177/4388)
  ville                [███████████████████░] 100.0%  (4386/4388)
  contrat              [██████████████████░░]  94.1%  (4130/4388)
  date_publication     [███████████████████░]  99.9%  (4384/4388)
  competences          [███████████████████░]  99.9%  (4384/4388)

── Top villes ──
ville
Dakar          4385
Saint-Louis       1
Name: count, dtype: int64

── Répartition contrats ──
contrat
CDD              2608
Stage             499
CDI               454
Intérim           230
Consultant        215
Freelance          75
Temps Partiel      44
Alternance          4
Bénévolat           1
Name: count, dtype: int64

── Répartition par source ──
source
goafricaonline.com    4167
senjob.com             221
Name: count, dtype: in

In [ ]:
# ── Aperçu du dataframe ───────────────────────────────────────────────────────
print(df[cols_cibles].head(10).to_string())

                                             titre     entreprise  ville     contrat date_publication                                                                           competences
0      Dev Full Stack Expérimenté – Java / Angular                 Dakar  Consultant       2026-04-22  Java, SQL, Angular, Azure, GCP, Docker, Kubernetes, Agile, Scrum, Finance, Recherche
1                            Chauffeurs de pick-up                 Dakar       Stage       2026-04-17                                                      Comptabilité, Finance, Recherche
2             Dev Front-End Expérimenté – React.js                 Dakar         CDI       2026-04-22                                       JavaScript, React, Git, Agile, Scrum, Recherche
3                Développeur Fullstack Java/Vue js                 Dakar         CDI       2026-04-30                  Java, SQL, Docker, Kubernetes, Agile, Scrum, Comptabilité, Recherche
4                          Agents de climatisation  AMD Corp

In [ ]:
# ── Complétion intelligente des valeurs manquantes ────────────────────────────

print("── Taux avant complétion ──")
for col in ["entreprise", "ville", "contrat", "date_publication", "competences"]:
    n_vide = df[col].replace("", pd.NA).isna().sum()
    print(f"  {col:<20} : {n_vide} valeurs manquantes")

# ── 1. VILLE ──────────────────────────────────────────────────────────────────
# Extraire depuis le titre ou le lien si encore vide
def ville_depuis_titre_lien(row):
    if row["ville"] and str(row["ville"]).strip():
        return row["ville"]
    # Chercher dans le titre
    for ville in VILLES_SENEGAL:
        if re.search(rf"\b{ville}\b", str(row["titre"]), re.IGNORECASE):
            return ville.title()
    # Chercher dans le lien
    for ville in VILLES_SENEGAL:
        if ville.lower() in str(row["lien"]).lower():
            return ville.title()
    # Défaut : Dakar (capitale économique, ~70% des offres)
    return "Dakar"

df["ville"] = df.apply(ville_depuis_titre_lien, axis=1)

# ── 2. CONTRAT ────────────────────────────────────────────────────────────────
# Inférer depuis le titre
def contrat_depuis_titre(row):
    if row["contrat"] and str(row["contrat"]).strip():
        return row["contrat"]
    titre = str(row["titre"]).lower()
    if any(k in titre for k in ["stage", "intern", "stagiaire"]):
        return "Stage"
    if any(k in titre for k in ["freelance", "consultant", "prestation"]):
        return "Freelance"
    if any(k in titre for k in ["temps partiel", "mi-temps", "partiel"]):
        return "Temps partiel"
    if any(k in titre for k in ["alternance", "apprenti"]):
        return "Alternance"
    if "cdd" in titre:
        return "CDD"
    # Défaut : CDI (contrat le plus courant pour les offres publiées)
    return "CDI"

df["contrat"] = df.apply(contrat_depuis_titre, axis=1)

# ── 3. ENTREPRISE ─────────────────────────────────────────────────────────────
# Extraire depuis le domaine du lien ou marquer "Confidentiel"
def entreprise_depuis_lien(row):
    if row["entreprise"] and str(row["entreprise"]).strip():
        return row["entreprise"]
    lien = str(row["lien"])
    # Essayer d'extraire le sous-domaine ou un segment du lien
    m = re.search(r"company[/-]([a-z0-9\-]+)", lien, re.IGNORECASE)
    if m:
        return m.group(1).replace("-", " ").title()
    m = re.search(r"employer[/-]([a-z0-9\-]+)", lien, re.IGNORECASE)
    if m:
        return m.group(1).replace("-", " ").title()
    return "Confidentiel"

df["entreprise"] = df.apply(entreprise_depuis_lien, axis=1)

# ── 4. DATE DE PUBLICATION ────────────────────────────────────────────────────
# Compléter les NaT avec la date du jour (offre supposée récente)
from datetime import datetime
aujourd_hui = pd.Timestamp(datetime.now().date())
df["date_publication"] = df["date_publication"].fillna(aujourd_hui)
# Formater proprement
df["date_publication"] = pd.to_datetime(df["date_publication"]).dt.strftime("%d/%m/%Y")

# ── 5. COMPÉTENCES ────────────────────────────────────────────────────────────
# Si vide, extraire depuis le titre seul
def competences_depuis_titre(row):
    if row["competences"] and str(row["competences"]).strip():
        return row["competences"]
    return extraire_competences(str(row["titre"]))

df["competences"] = df.apply(competences_depuis_titre, axis=1)
# Si toujours vide après le titre → "Non précisées"
df["competences"] = df["competences"].replace("", "Non précisées")

# ── Rapport après complétion ──────────────────────────────────────────────────
print("\n── Taux après complétion ──")
for col in ["entreprise", "ville", "contrat", "date_publication", "competences"]:
    n_vide = df[col].replace("", pd.NA).isna().sum()
    pct_rempli = (1 - n_vide / len(df)) * 100
    bar = "█" * int(pct_rempli // 5) + "░" * (20 - int(pct_rempli // 5))
    print(f"  {col:<20} [{bar}] {pct_rempli:5.1f}%")

print(f"\n✅ Complétion terminée — {len(df)} offres prêtes à exporter")

── Taux avant complétion ──
  entreprise           : 211 valeurs manquantes
  ville                : 2 valeurs manquantes
  contrat              : 258 valeurs manquantes
  date_publication     : 4 valeurs manquantes
  competences          : 4 valeurs manquantes

── Taux après complétion ──
  entreprise           [████████████████████] 100.0%
  ville                [████████████████████] 100.0%
  contrat              [████████████████████] 100.0%
  date_publication     [████████████████████] 100.0%
  competences          [████████████████████] 100.0%

✅ Complétion terminée — 4388 offres prêtes à exporter


In [ ]:
# ── Sauvegarde CSV ────────────────────────────────────────────────────────────
nom_fichier = f"offres_emploi_senegal_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

df.to_csv(nom_fichier, index=False, encoding="utf-8-sig")
print(f"✅ Sauvegardé : {nom_fichier}")
print(f"   Taille : {df.shape[0]} lignes × {df.shape[1]} colonnes")

✅ Sauvegardé : offres_emploi_senegal_20260417_1332.csv
   Taille : 4388 lignes × 8 colonnes


In [ ]:
!pip install openpyxl -q

In [ ]:
# ── Sauvegarde & Téléchargement ───────────────────────────────────────────────
from datetime import datetime
from google.colab import files

nom_fichier = f"offres_emploi_senegal_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

# Sauvegarde Excel (meilleur rendu que CSV pour Excel)
with pd.ExcelWriter(nom_fichier, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Offres")

    # ── Mise en forme automatique ──
    ws = writer.sheets["Offres"]
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    # En-têtes en gras + fond bleu
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # Largeur automatique des colonnes
    for col_idx, col in enumerate(df.columns, 1):
        max_len = max(df[col].astype(str).map(len).max(), len(col)) + 4
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len, 60)

    # Figer la première ligne
    ws.freeze_panes = "A2"

print(f"✅ Fichier Excel créé : {nom_fichier}")
print(f"   {df.shape[0]} lignes × {df.shape[1]} colonnes")

# ── Téléchargement direct dans le navigateur ──
files.download(nom_fichier)
print("📥 Téléchargement lancé !")

✅ Fichier Excel créé : offres_emploi_senegal_20260417_1332.xlsx
   4388 lignes × 8 colonnes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Téléchargement lancé !


In [ ]:
df.head()

,titre,entreprise,ville,contrat,date_publication,competences,lien,source
0,Dev Full Stack Expérimenté – Java / Angular,Confidentiel,Dakar,Consultant,22/04/2026,"Java, SQL, Angular, Azure, GCP, Docker, Kubern...",https://senjob.com/sn/jobseekers/dev-full-stac...,senjob.com
1,Chauffeurs de pick-up,Confidentiel,Dakar,Stage,17/04/2026,"Comptabilité, Finance, Recherche",https://senjob.com/sn/jobseekers/chauffeurs-de...,senjob.com
2,Dev Front-End Expérimenté – React.js,Confidentiel,Dakar,CDI,22/04/2026,"JavaScript, React, Git, Agile, Scrum, Recherche",https://senjob.com/sn/jobseekers/dev-front-end...,senjob.com
3,Développeur Fullstack Java/Vue js,Confidentiel,Dakar,CDI,30/04/2026,"Java, SQL, Docker, Kubernetes, Agile, Scrum, C...",https://senjob.com/sn/jobseekers/développeur-f...,senjob.com
4,Agents de climatisation,AMD Corporate,Dakar,CDI,01/05/2026,"Java, Finance, Recherche",https://senjob.com/sn/jobseekers/agents-de-cli...,senjob.com


In [ ]:
# ── Imports NLP ───────────────────────────────────────────────────────────────
!pip install unidecode -q
import spacy
import unicodedata
from unidecode import unidecode
from collections import Counter

# Download the French spaCy model if it's not already installed
!python -m spacy download fr_core_news_md

nlp = spacy.load('fr_core_news_md')
print('✅ Modèle spaCy chargé')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 13.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Modèle spaCy chargé


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ÉTAPE 1 — NORMALISATION DES MÉTIERS
# ══════════════════════════════════════════════════════════════════════════════

# Dictionnaire de normalisation : variantes → forme canonique
METIERS_NORMALISATION = {
    # Informatique / Tech
    'developpeur': 'Développeur',
    'developer': 'Développeur',
    'dev ': 'Développeur',
    'ingenieur logiciel': 'Ingénieur Logiciel',
    'software engineer': 'Ingénieur Logiciel',
    'ingenieur informatique': 'Ingénieur Informatique',
    'data scientist': 'Data Scientist',
    'data analyst': 'Data Analyst',
    'analyste de donnees': 'Data Analyst',
    'analyste donnees': 'Data Analyst',
    'data engineer': 'Data Engineer',
    'machine learning': 'Ingénieur ML',
    'administrateur systeme': 'Administrateur Système',
    'sysadmin': 'Administrateur Système',
    'devops': 'DevOps',
    'chef de projet': 'Chef de Projet',
    'project manager': 'Chef de Projet',
    'product manager': 'Product Manager',
    'product owner': 'Product Owner',
    'scrum master': 'Scrum Master',
    'ui ux': 'Designer UI/UX',
    'ux designer': 'Designer UI/UX',
    'graphiste': 'Graphiste',
    'web designer': 'Web Designer',
    'webmaster': 'Webmaster',
    'technicien informatique': 'Technicien Informatique',
    'support technique': 'Technicien Support',
    'help desk': 'Technicien Support',
    'cybersecurite': 'Expert Cybersécurité',
    'securite informatique': 'Expert Cybersécurité',
    # Finance / Compta
    'comptable': 'Comptable',
    'directeur financier': 'Directeur Financier',
    'daf': 'Directeur Administratif et Financier',
    'directeur administratif': 'Directeur Administratif et Financier',
    'auditeur': 'Auditeur',
    'controleur de gestion': 'Contrôleur de Gestion',
    'tresorier': 'Trésorier',
    'fiscaliste': 'Fiscaliste',
    'analyste financier': 'Analyste Financier',
    # Commerce / Marketing
    'commercial': 'Commercial',
    'charge de clientele': 'Chargé de Clientèle',
    'responsable commercial': 'Responsable Commercial',
    'directeur commercial': 'Directeur Commercial',
    'charge de marketing': 'Chargé Marketing',
    'responsable marketing': 'Responsable Marketing',
    'community manager': 'Community Manager',
    'charge de communication': 'Chargé de Communication',
    'chef de produit': 'Chef de Produit',
    'key account': 'Key Account Manager',
    'business developer': 'Business Developer',
    'business development': 'Business Developer',
    'responsable des ventes': 'Responsable des Ventes',
    # RH
    'responsable rh': 'Responsable RH',
    'drh': 'Directeur des Ressources Humaines',
    'directeur des ressources humaines': 'Directeur des Ressources Humaines',
    'charge de recrutement': 'Chargé de Recrutement',
    'recruteur': 'Chargé de Recrutement',
    'gestionnaire rh': 'Gestionnaire RH',
    'charge de formation': 'Chargé de Formation',
    # Autres
    'juriste': 'Juriste',
    'avocat': 'Avocat',
    'medecin': 'Médecin',
    'infirmier': 'Infirmier',
    'pharmacien': 'Pharmacien',
    'enseignant': 'Enseignant',
    'formateur': 'Formateur',
    'chauffeur': 'Chauffeur',
    'logisticien': 'Logisticien',
    'responsable logistique': 'Responsable Logistique',
    'supply chain': 'Supply Chain Manager',
    'secretaire': 'Secrétaire',
    'assistante de direction': 'Assistant(e) de Direction',
    'office manager': 'Office Manager',
    'charge de projet': 'Chargé de Projet',
    'ingenieur': 'Ingénieur',
    'technicien': 'Technicien',
    'agent': 'Agent',
    'responsable': 'Responsable',
    'directeur': 'Directeur',
    'stagiaire': 'Stagiaire',
}

def normaliser_titre(titre):
    """Normalise un intitulé de poste vers une forme canonique."""
    if not titre or str(titre).strip() == '':
        return titre

    titre_clean = str(titre).strip()

    # Version simplifiée pour la comparaison (sans accents, minuscules)
    titre_cle = unidecode(titre_clean).lower()
    titre_cle = re.sub(r'[^a-z0-9\s]', ' ', titre_cle)
    titre_cle = re.sub(r'\s+', ' ', titre_cle).strip()

    # Chercher la correspondance la plus longue en premier
    meilleur_match = None
    meilleur_len = 0
    for pattern, forme_canon in METIERS_NORMALISATION.items():
        if pattern in titre_cle and len(pattern) > meilleur_len:
            meilleur_match = forme_canon
            meilleur_len = len(pattern)

    if meilleur_match:
        # Conserver les infos de spécialisation (ex: "Développeur Python Senior")
        suffixes = []
        for spec in ['senior', 'junior', 'confirmé', 'débutant', 'expert', 'lead', 'manager',
                     'python', 'java', 'react', 'angular', 'mobile', 'web', 'full stack',
                     'backend', 'frontend', 'fullstack', 'bi', 'erp', 'sap']:
            if spec in titre_cle:
                suffixes.append(spec.title())
        if suffixes:
            return f"{meilleur_match} ({', '.join(suffixes)})"
        return meilleur_match

    # Fallback : nettoyer et mettre en title case
    titre_clean = re.sub(r'\s+', ' ', titre_clean).strip()
    return titre_clean.title()


# Application
df['titre_original'] = df['titre'].copy()          # conserver l'original
df['titre_normalise'] = df['titre'].apply(normaliser_titre)

print('✅ Normalisation des métiers terminée')
print(f'   Exemple de normalisations :')
masque = df['titre_normalise'] != df['titre_original']
print(df[masque][['titre_original', 'titre_normalise']].head(10).to_string(index=False))

✅ Normalisation des métiers terminée
   Exemple de normalisations :
                                 titre_original                                 titre_normalise
    Dev Full Stack Expérimenté – Java / Angular         Développeur (Java, Angular, Full Stack)
                          Chauffeurs de pick-up                                       Chauffeur
           Dev Front-End Expérimenté – React.js                             Développeur (React)
              Développeur Fullstack Java/Vue js                   Développeur (Java, Fullstack)
                        Agents de climatisation                                           Agent
    Conseiller(e) en Chaine d’Approvisionnement     Conseiller(E) En Chaine D’Approvisionnement
           Assistant aux opérations financières            Assistant Aux Opérations Financières
Télévendeurs en conquête mobile/marché français Télévendeurs En Conquête Mobile/Marché Français
              Community Manager / Infographiste                     

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ÉTAPE 2 — EXTRACTION DES COMPÉTENCES AVEC spaCy (NLP)
# ══════════════════════════════════════════════════════════════════════════════

# Taxonomie étendue des compétences avec catégories
TAXONOMIE_COMPETENCES = {
    'Langages de programmation': [
        'Python', 'Java', 'JavaScript', 'TypeScript', 'PHP', 'C', 'C++', 'C#',
        'R', 'Kotlin', 'Swift', 'Dart', 'Go', 'Ruby', 'Scala', 'Rust', 'MATLAB',
        'VBA', 'Bash', 'Shell', 'Perl',
    ],
    'Web & Frameworks': [
        'React', 'Angular', 'Vue.js', 'Node.js', 'Django', 'Flask', 'FastAPI',
        'Laravel', 'Symfony', 'Spring', 'Express', 'Next.js', 'Nuxt.js',
        'Bootstrap', 'Tailwind', 'HTML', 'CSS', 'REST', 'GraphQL', 'API',
    ],
    'Data & IA': [
        'SQL', 'MySQL', 'PostgreSQL', 'MongoDB', 'Oracle', 'SQLite', 'Redis',
        'Power BI', 'Tableau', 'Excel', 'SPSS', 'SAS', 'Stata',
        'Machine Learning', 'Deep Learning', 'NLP', 'TensorFlow', 'PyTorch',
        'Scikit-learn', 'Pandas', 'NumPy', 'Hadoop', 'Spark', 'Kafka',
        'ETL', 'Data Warehouse', 'Big Data', 'BI', 'Looker',
    ],
    'Cloud & DevOps': [
        'AWS', 'Azure', 'GCP', 'Docker', 'Kubernetes', 'Git', 'GitHub',
        'GitLab', 'CI/CD', 'Jenkins', 'Terraform', 'Ansible', 'Linux',
        'Unix', 'Windows Server', 'VMware',
    ],
    'Design': [
        'Photoshop', 'Illustrator', 'Figma', 'InDesign', 'Canva',
        'After Effects', 'Premiere Pro', 'Sketch', 'AutoCAD',
    ],
    'Gestion & Management': [
        'Gestion de projet', 'Agile', 'Scrum', 'Kanban', 'PRINCE2', 'PMP',
        'ITIL', 'SAP', 'ERP', 'CRM', 'Salesforce', 'HubSpot', 'Odoo',
        'Microsoft Project', 'JIRA', 'Trello', 'Notion',
    ],
    'Finance & Compta': [
        'Comptabilité', 'Audit', 'Fiscalité', 'SYSCOHADA', 'OHADA',
        'Contrôle de gestion', 'Budget', 'Trésorerie', 'Finance',
        'Reporting', 'Analyse financière', 'IFRS', 'Sage', 'CEGID',
    ],
    'Marketing & Com': [
        'Marketing digital', 'SEO', 'SEM', 'Google Ads', 'Facebook Ads',
        'Community management', 'Content marketing', 'Email marketing',
        'Rédaction', 'Copywriting', 'Relations publiques', 'Communication',
        'Google Analytics', 'CMS', 'WordPress',
    ],
    'Langues & Soft Skills': [
        'Anglais', 'Français', 'Wolof', 'Espagnol', 'Arabe', 'Portugais',
        'Leadership', 'Management', 'Négociation', 'Communication',
        'Travail en équipe', 'Autonomie', 'Rigueur', 'Créativité',
        'Adaptabilité', 'Organisation', 'Analyse', 'Synthèse',
    ],
}

# Index plat : terme → catégorie
INDEX_COMPETENCES = {}
for categorie, termes in TAXONOMIE_COMPETENCES.items():
    for terme in termes:
        INDEX_COMPETENCES[unidecode(terme).lower()] = (terme, categorie)


def extraire_competences_nlp(texte):
    """
    Extraction NLP en 3 couches :
      1. Matching exact depuis la taxonomie
      2. Reconnaissance d'entités spaCy (ORG, MISC)
      3. Extraction de noms composés (chunks noun)
    Retourne un dict {terme: catégorie}
    """
    if not texte or str(texte).strip() in ['', 'Non précisées']:
        return {}

    texte_str = str(texte)
    texte_cle = unidecode(texte_str).lower()
    trouvees = {}

    # ── Couche 1 : matching exact taxonomie ──
    for cle, (terme, categorie) in INDEX_COMPETENCES.items():
        # Chercher le terme comme mot entier
        pattern = rf'\b{re.escape(cle)}\b'
        if re.search(pattern, texte_cle):
            trouvees[terme] = categorie

    # ── Couche 2 : spaCy NER (entités organisationnelles = souvent des technos) ──
    doc = nlp(texte_str[:2000])  # Limiter à 2000 chars pour la perf
    for ent in doc.ents:
        if ent.label_ in ('ORG', 'MISC', 'PRODUCT'):
            val = ent.text.strip()
            val_cle = unidecode(val).lower()
            if val_cle in INDEX_COMPETENCES:
                terme, cat = INDEX_COMPETENCES[val_cle]
                trouvees[terme] = cat
            elif len(val) >= 2 and not val.lower() in ['le', 'la', 'les', 'un', 'une', 'des']:
                # Entité non connue mais potentiellement pertinente
                trouvees[val] = 'Autre'

    # ── Couche 3 : noun chunks (ex: "gestion de projet", "analyse financière") ──
    for chunk in doc.noun_chunks:
        val = chunk.text.strip()
        val_cle = unidecode(val).lower()
        if val_cle in INDEX_COMPETENCES:
            terme, cat = INDEX_COMPETENCES[val_cle]
            trouvees[terme] = cat

    return trouvees


def formater_competences(comp_dict):
    """Formate le dict compétences → string lisible."""
    if not comp_dict:
        return 'Non précisées'
    return ', '.join(sorted(comp_dict.keys()))


def formater_categories(comp_dict):
    """Retourne les catégories uniques des compétences trouvées."""
    if not comp_dict:
        return ''
    cats = sorted(set(comp_dict.values()) - {'Autre'})
    return ', '.join(cats)


# ── Application sur le dataframe ──
# Combiner le titre + les compétences déjà extraites pour maximiser la détection
df['texte_nlp'] = df['titre_normalise'].fillna('') + ' ' + df['competences'].fillna('')

print('Extraction NLP en cours (peut prendre 1-2 min)...')
comp_dicts = df['texte_nlp'].apply(extraire_competences_nlp)

df['competences_nlp']   = comp_dicts.apply(formater_competences)
df['categories_comp']   = comp_dicts.apply(formater_categories)

# Stats
n_avec_comp = (df['competences_nlp'] != 'Non précisées').sum()
print(f'\n✅ Extraction NLP terminée')
print(f'   Offres avec compétences identifiées : {n_avec_comp}/{len(df)} ({n_avec_comp/len(df)*100:.1f}%)')

# Top 15 compétences les plus demandées
toutes_comp = []
for d in comp_dicts:
    toutes_comp.extend(d.keys())
top_comp = Counter(toutes_comp).most_common(15)
print(f'\n── Top 15 compétences les plus demandées ──')
for comp, n in top_comp:
    bar = '█' * int(n / max(c for _, c in top_comp) * 20)
    print(f'  {comp:<30} {bar} ({n})')

Extraction NLP en cours (peut prendre 1-2 min)...

✅ Extraction NLP terminée
   Offres avec compétences identifiées : 4386/4388 (100.0%)

── Top 15 compétences les plus demandées ──
  Français                       ████████████████████ (4224)
  Communication                  ███████████████████ (4156)
  Excel                          ████████████████ (3522)
  Anglais                        ████████████████ (3449)
  Comptabilité                   ██████████████ (3084)
  Négociation                    ███████████ (2530)
  Gestion de projet              ██████████ (2249)
  Wolof                          ██████████ (2223)
  Finance                        ██████████ (2170)
  Statistiques                   ███████ (1624)
  Audit                          ██████ (1347)
  Tableau                        █████ (1193)
  Négociation, Recherche         █████ (1113)
  SAP                            ████ (1053)
  Analyse                        ████ (988)
